# 02 多项式插值

本节讨论第二章的主要全局多项式工具：

* 拉格朗日插值（Lagrange interpolation）的构造来源；
* 牛顿差商；
* 等距节点上的 Runge 现象；
* 切比雪夫节点带来的改进。

拉格朗日插值的核心思想是构造一组“开关函数”：第 $j$ 个基函数只在第 $j$ 个节点处取 1，在其他节点处取 0。这样就可以把每个节点值安装到对应节点上。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    chebyshev_nodes,
    divided_difference_table,
    lagrange_interpolate,
    newton_divided_differences,
    newton_interpolate,
)


## 从两个点开始

已知两个点

$$
(x_0,f_0),\qquad (x_1,f_1),
$$

通过这两个点的直线可以写成

$$
p_1(x)=f_0\frac{x-x_1}{x_0-x_1}
+f_1\frac{x-x_0}{x_1-x_0}.
$$

这个写法的关键在于

$$
\frac{x-x_1}{x_0-x_1}
$$

在 $x=x_0$ 处等于 1，在 $x=x_1$ 处等于 0；而

$$
\frac{x-x_0}{x_1-x_0}
$$

在 $x=x_0$ 处等于 0，在 $x=x_1$ 处等于 1。因此这两个因子就像两个开关，分别控制 $f_0$ 和 $f_1$ 是否被选中。


In [ ]:
x0, x1 = 0.0, 2.0
f0, f1 = 1.0, 3.0
x_line = np.linspace(-0.2, 2.2, 300)

ell0 = (x_line - x1) / (x0 - x1)
ell1 = (x_line - x0) / (x1 - x0)
p1 = f0 * ell0 + f1 * ell1

plt.figure(figsize=(8, 4.5))
plt.plot(x_line, ell0, label=r"$\ell_0(x)$")
plt.plot(x_line, ell1, label=r"$\ell_1(x)$")
plt.plot(x_line, p1, label=r"$p_1(x)$")
plt.scatter([x0, x1], [f0, f1], color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("两点插值中的开关函数")
plt.legend()
plt.tight_layout()
plt.show()


## 一般情形：构造第 j 个基函数

现在给定 $N+1$ 个互异节点

$$
x_0,x_1,\dots,x_N.
$$

希望构造第 $j$ 个基函数 $\ell_j(x)$，使得

$$
\ell_j(x_i)=
\begin{cases}
1, & i=j,\\
0, & i\neq j.
\end{cases}
$$

为了让它在所有其他节点 $x_i,\ i\neq j$ 处为零，最自然的做法是让它含有因子

$$
\prod_{\substack{m=0\\m\neq j}}^N (x-x_m).
$$

这样只要 $x=x_i$ 且 $i\neq j$，其中就有一个因子变成 0。接下来再让它在 $x_j$ 处等于 1，需要除以

$$
\prod_{\substack{m=0\\m\neq j}}^N (x_j-x_m).
$$

因此得到拉格朗日基函数

$$
\boxed{
\ell_j(x)=
\prod_{\substack{m=0\\m\neq j}}^N
\frac{x-x_m}{x_j-x_m}
}.
$$


In [ ]:
def lagrange_basis(nodes, basis_index, x_eval):
    values = np.ones_like(x_eval, dtype=float)
    xj = nodes[basis_index]
    for m, xm in enumerate(nodes):
        if m != basis_index:
            values *= (x_eval - xm) / (xj - xm)
    return values

nodes = np.array([-1.0, -0.3, 0.4, 1.0])
x_plot = np.linspace(-1.1, 1.1, 500)

plt.figure(figsize=(8, 4.5))
for j in range(nodes.size):
    plt.plot(x_plot, lagrange_basis(nodes, j, x_plot), label=rf"$\ell_{j}(x)$")
plt.scatter(nodes, np.zeros_like(nodes), color="black", zorder=3)
plt.xlabel("x")
plt.ylabel("基函数值")
plt.title("拉格朗日基函数：每个基函数负责一个节点")
plt.legend()
plt.tight_layout()
plt.show()


## 拼出插值多项式

若 $\ell_j(x_i)=\delta_{ij}$，则

$$
p_N(x)=\sum_{j=0}^{N} f_j\ell_j(x)
$$

在节点 $x_i$ 上满足

$$
p_N(x_i)=
\sum_{j=0}^{N} f_j\ell_j(x_i)
=
\sum_{j=0}^{N} f_j\delta_{ij}
=f_i.
$$

所以拉格朗日插值多项式为

$$
\boxed{
p_N(x)=
\sum_{j=0}^{N}
f_j
\prod_{\substack{m=0\\m\neq j}}^{N}
\frac{x-x_m}{x_j-x_m}
}.
$$


## 三点例子

给定三个点

$$
(x_0,f_0),\qquad (x_1,f_1),\qquad (x_2,f_2),
$$

二次插值多项式为

$$
p_2(x)=f_0\ell_0(x)+f_1\ell_1(x)+f_2\ell_2(x),
$$

其中

$$
\ell_0(x)=
\frac{(x-x_1)(x-x_2)}
{(x_0-x_1)(x_0-x_2)},
$$

$$
\ell_1(x)=
\frac{(x-x_0)(x-x_2)}
{(x_1-x_0)(x_1-x_2)},
$$

$$
\ell_2(x)=
\frac{(x-x_0)(x-x_1)}
{(x_2-x_0)(x_2-x_1)}.
$$

这些基函数在节点上的取值正好组成单位矩阵，因此插值多项式严格通过三个给定点。


In [ ]:
nodes_three = np.array([-1.0, 0.2, 1.0])
values_three = np.array([1.0, -0.4, 0.6])
x_three = np.linspace(-1.1, 1.1, 400)

basis_at_nodes = np.array([
    [lagrange_basis(nodes_three, j, xi) for j in range(nodes_three.size)]
    for xi in nodes_three
])
print("三点例子中，基函数在节点上的取值矩阵：")
print(np.round(basis_at_nodes, 6))

p2 = lagrange_interpolate(nodes_three, values_three, x_three)

plt.figure(figsize=(8, 4.5))
for j in range(nodes_three.size):
    plt.plot(x_three, lagrange_basis(nodes_three, j, x_three), "--", alpha=0.75, label=rf"$\ell_{j}(x)$")
plt.plot(x_three, p2, color="black", linewidth=2, label=r"$p_2(x)$")
plt.scatter(nodes_three, values_three, color="red", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("三点拉格朗日插值示例")
plt.legend()
plt.tight_layout()
plt.show()


## 为什么插值多项式唯一？

假设 $p_N(x)$ 和 $q_N(x)$ 都是不超过 $N$ 次的多项式，并且都通过这 $N+1$ 个节点。令

$$
r(x)=p_N(x)-q_N(x).
$$

那么 $r(x)$ 仍然是不超过 $N$ 次的多项式，而且

$$
r(x_j)=0,\qquad j=0,1,\dots,N.
$$

也就是说，$r(x)$ 有 $N+1$ 个互异零点。一个不超过 $N$ 次的非零多项式最多只有 $N$ 个零点，所以只能有 $r(x)\equiv 0$。因此 $p_N(x)=q_N(x)$，插值多项式唯一。


## 与切比雪夫微分矩阵的关系

在切比雪夫配置法中，常用节点为

$$
x_j=\cos\frac{j\pi}{N},\qquad j=0,1,\dots,N.
$$

先用这些节点构造拉格朗日插值

$$
p_N(x)=\sum_{j=0}^N f(x_j)\ell_j(x),
$$

再对插值多项式求导：

$$
p_N'(x_i)=\sum_{j=0}^N f(x_j)\ell_j'(x_i).
$$

于是可以定义

$$
D_{ij}=\ell_j'(x_i),
$$

这就是切比雪夫微分矩阵的基本来源。逻辑链条是

$$
\boxed{
\text{节点函数值}
\rightarrow
\text{拉格朗日插值多项式}
\rightarrow
\text{对插值多项式求导}
\rightarrow
\text{微分矩阵}
}.
$$

本章只说明这个来源，具体的矩阵公式、稳定性和谱精度会留到后续“谱方法”章节。


## 牛顿差商

牛顿插值（Newton interpolation）写作

$$
p_n(x)=c_0+c_1(x-x_0)+c_2(x-x_0)(x-x_1)+\cdots.
$$

系数来自牛顿差商表的第一行。这个形式在逐步加入新节点时很方便，因为已有的低阶系数可以继续使用。


In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 0.0, 4.0])
x_eval = np.linspace(0.0, 3.0, 300)

lagrange_values = lagrange_interpolate(x, y, x_eval)
newton_nodes, coefficients = newton_divided_differences(x, y)
newton_values = newton_interpolate(newton_nodes, coefficients, x_eval)
_, table = divided_difference_table(x, y)

print("牛顿插值系数:", coefficients)
print("差商表第一行:", table[0])

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, lagrange_values, label="拉格朗日形式")
plt.plot(x_eval, newton_values, "--", label="牛顿形式")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("p(x)")
plt.title("拉格朗日形式与牛顿形式表示同一个多项式")
plt.legend()
plt.tight_layout()
plt.show()


## Runge 现象

插值误差公式中含有节点乘积

$$
\prod_{i=0}^{n}(x-x_i).
$$

在等距节点上做高次插值时，这一项可能在区间端点附近变大，从而导致明显振荡。Runge 函数是检验这一问题的经典例子。


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_plot = np.linspace(-1.0, 1.0, 1000)
y_true = runge(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, n in zip(axes, [9, 17]):
    x_equal = np.linspace(-1.0, 1.0, n)
    y_equal = runge(x_equal)
    y_interp = lagrange_interpolate(x_equal, y_equal, x_plot)
    ax.plot(x_plot, y_true, color="black", label="Runge 函数")
    ax.plot(x_plot, y_interp, label=f"等距节点，节点数={n}")
    ax.scatter(x_equal, y_equal, s=18, zorder=3)
    ax.set_xlabel("x")
    ax.set_title(f"{n - 1} 次插值多项式")
axes[0].set_ylabel("函数值")
axes[0].legend()
fig.suptitle("等距节点上的端点振荡")
fig.tight_layout()
plt.show()


## 切比雪夫节点

切比雪夫节点（Chebyshev nodes）在区间端点附近更密集，可以减小节点乘积的增长。它不能解决所有插值问题，但在光滑函数的全局多项式插值中非常重要。


In [ ]:
node_counts = [9, 17, 25]
for n in node_counts:
    equal_nodes = np.linspace(-1.0, 1.0, n)
    cheb_nodes = chebyshev_nodes(-1.0, 1.0, n)

    equal_error = np.max(np.abs(lagrange_interpolate(equal_nodes, runge(equal_nodes), x_plot) - y_true))
    cheb_error = np.max(np.abs(lagrange_interpolate(cheb_nodes, runge(cheb_nodes), x_plot) - y_true))
    print(f"节点数={n:2d}, 等距节点误差={equal_error:10.3e}, 切比雪夫节点误差={cheb_error:10.3e}")

n = 17
x_equal = np.linspace(-1.0, 1.0, n)
x_cheb = chebyshev_nodes(-1.0, 1.0, n)

plt.figure(figsize=(8, 4.5))
plt.plot(x_plot, y_true, color="black", label="Runge 函数")
plt.plot(x_plot, lagrange_interpolate(x_equal, runge(x_equal), x_plot), label="等距节点")
plt.plot(x_plot, lagrange_interpolate(x_cheb, runge(x_cheb), x_plot), label="切比雪夫节点")
plt.scatter(x_equal, runge(x_equal), s=18, alpha=0.7)
plt.scatter(x_cheb, runge(x_cheb), s=18, alpha=0.7)
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("切比雪夫节点可以减弱端点振荡")
plt.legend()
plt.tight_layout()
plt.show()


## 小结

* 拉格朗日形式直观，适合理解插值基函数。
* 牛顿形式适合增量添加节点和构造差商表。
* 等距节点上的高次全局插值可能严重失效。
* 对有界区间上的光滑函数，切比雪夫节点是重要的稳定化工具。
